# 04 · Deep Learning: autoencoder
[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessicapmartins/segmentacao-clientes-olist/blob/main/notebooks/04_deep_learning_autoencoder.ipynb)

**Ideia:** uma rede neural aprende a comprimir as variáveis do cliente em 2 números e a reconstruí-las. Esses 2 números (o "resumo" da rede) podem ser clusterizados e também viram um gráfico 2D dos clientes.

**Pergunta:** os grupos encontrados assim são diferentes dos do K-Means? Vale a complexidade?

*Não precisa de GPU: são só 3 variáveis, e a CPU do Colab treina em cerca de 1 minuto.*

In [ ]:
import sqlite3, pandas as pd

try:                                    # no Colab: lê o banco do seu Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    DADOS = '/content/drive/MyDrive/03_Portfolio/Projetos_ML/01_segmentacao-clientes-olist/dados'
except ModuleNotFoundError:             # fora do Colab: pasta dados/ do projeto
    DADOS = '../dados'

con = sqlite3.connect(f'{DADOS}/olist.db')

def q(sql):
    """Roda uma consulta SQL no banco da Olist e devolve um DataFrame."""
    return pd.read_sql(sql, con)

# Este notebook usa a tabela 'segmentos', criada no fim do notebook 03
tabelas = q("SELECT name FROM sqlite_master WHERE type='table'")['name'].tolist()
print('Banco em uso:', DADOS)
assert 'segmentos' in tabelas, 'Tabela segmentos não encontrada: rode o notebook 03 até o fim antes deste.'

import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import keras
from keras import layers

keras.utils.set_random_seed(42)          # resultados reproduzíveis
sns.set_theme(style='whitegrid')

## 1. Dados padronizados
As mesmas três variáveis do notebook 03 (recência, log da frequência, log do valor), padronizadas.

In [ ]:
seg = q('SELECT * FROM segmentos')       # um cliente por linha, já com o segmento do K-Means

X = pd.DataFrame({
    'recencia':       seg['recencia'],
    'frequencia_log': np.log1p(seg['frequencia']),
    'valor_log':      np.log1p(seg['valor']),
})
Xs = StandardScaler().fit_transform(X).astype('float32')
print(Xs.shape)

## 2. Montar o autoencoder
Entrada (3) → Dense(8) → **Dense(2), o gargalo** → Dense(8) → saída (3). A rede é treinada para devolver a própria entrada; para conseguir, precisa resumir cada cliente em 2 números.

In [ ]:
entrada = keras.Input(shape=(Xs.shape[1],))
x       = layers.Dense(8, activation='relu')(entrada)
gargalo = layers.Dense(2, name='gargalo')(x)            # o "resumo" de 2 números
x       = layers.Dense(8, activation='relu')(gargalo)
saida   = layers.Dense(Xs.shape[1])(x)                  # reconstrução das 3 variáveis

autoencoder = keras.Model(entrada, saida)               # rede completa (treino)
encoder     = keras.Model(entrada, gargalo)             # só a metade que gera o resumo

autoencoder.compile(optimizer='adam', loss='mse')       # erro quadrático médio
autoencoder.summary()

## 3. Treinar e olhar a curva de perda
20% dos clientes ficam separados para validação. Se as duas curvas descem juntas e se estabilizam, não há sobreajuste.

In [ ]:
hist = autoencoder.fit(Xs, Xs, epochs=30, batch_size=512,
                       validation_split=0.2, verbose=0)

plt.figure(figsize=(8, 3.5))
plt.plot(hist.history['loss'], label='treino')
plt.plot(hist.history['val_loss'], label='validação')
plt.xlabel('época'); plt.ylabel('MSE'); plt.title('Curva de perda'); plt.legend()
plt.show()

print('Perda final, treino:', round(hist.history['loss'][-1], 4),
      '| validação:', round(hist.history['val_loss'][-1], 4))

## 4. Clusterizar o gargalo e comparar
K-Means com K = 4 nos 2 números do gargalo, comparado com os segmentos do notebook 03.

In [ ]:
Z = encoder.predict(Xs, verbose=0)                      # 2 números por cliente
seg['cluster_ae'] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(Z)

# Mapa 2D dos clientes, colorido pelo segmento do K-Means
amostra = np.random.RandomState(42).choice(len(Z), 8_000, replace=False)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=Z[amostra, 0], y=Z[amostra, 1], hue=seg['segmento'].values[amostra],
                hue_order=['Recorrentes', 'Alto valor', 'Novos de baixo valor', 'Inativos'],
                s=8, alpha=0.6)
plt.title('Clientes no espaço do gargalo (cor = segmento do K-Means)')
plt.xlabel('gargalo 1'); plt.ylabel('gargalo 2'); plt.legend(markerscale=2)
plt.show()

print('ARI segmentos K-Means × clusters do autoencoder:',
      round(adjusted_rand_score(seg['segmento'], seg['cluster_ae']), 3))
pd.crosstab(seg['segmento'], seg['cluster_ae'], rownames=['K-Means (nb 03)'], colnames=['Autoencoder'])

## Conclusão
**Como ler o resultado** (preencha depois de rodar):
- **ARI acima de 0,7:** o autoencoder encontrou praticamente os mesmos grupos que o K-Means. Com só 3 variáveis, a rede neural **não compensa a complexidade**; o ganho real dela é o mapa 2D, útil para visualizar.
- **ARI abaixo de 0,7:** olhe a tabela cruzada. Se os novos grupos tiverem um perfil de negócio claro, vale investigar; se misturarem perfis, o K-Means continua sendo a melhor escolha, por ser mais simples e explicável.

**Minha conclusão:** _ARI = ___. ..._